# Detection d'incoherences RCC8

Un reseau RCC8 est incoherent lorsqu'au moins une paire de regions ne possede plus aucune relation possible. Ce notebook montre comment le solveur detecte ces contradictions.

On distingue:

- les contradictions directes;
- les contradictions entre une relation et son inverse;
- les contradictions indirectes, visibles seulement apres propagation.

In [1]:
from rcc8.rcc8solver import RCC8Solver
from rcc8.relations import ALL_RELATIONS

RELATION_ORDER = ["DC", "EC", "PO", "EQ", "TPP", "NTPP", "TPPI", "NTPPI"]


def build_solver(regions):
    constraints = {}
    for i in regions:
        for j in regions:
            if i != j:
                constraints[(i, j)] = set(ALL_RELATIONS)
    return RCC8Solver(list(regions), constraints)


def fmt(relations):
    return "{" + ", ".join(r for r in RELATION_ORDER if r in relations) + "}"


def show_pair(solver, a, b):
    print(f"R({a},{b}) = {fmt(solver.R[(a, b)])}")
    print(f"R({b},{a}) = {fmt(solver.R[(b, a)])}")


def run_case(title, setup, expected_consistent):
    print("=" * 80)
    print(title)
    solver = build_solver(("A", "B", "C"))
    try:
        setup(solver)
        solver.pc2()
        print("Resultat: coherent")
        assert expected_consistent is True
        return solver
    except ValueError as error:
        print("Resultat: incoherent")
        print("Diagnostic:", error)
        assert expected_consistent is False
        return None

## 1. Cas coherent de reference

Avant les contradictions, on verifie un cas coherent simple:

- `A TPP B`;
- `B TPP C`;
- donc `A` est dans `C`.

In [2]:
def coherent_chain(solver):
    solver.add_constraint("A", "B", {"TPP"})
    solver.add_constraint("B", "C", {"TPP"})


solver_ok = run_case("Cas coherent: A TPP B, B TPP C", coherent_chain, True)
show_pair(solver_ok, "A", "C")

assert solver_ok.R[("A", "C")] == {"TPP", "NTPP"}

Cas coherent: A TPP B, B TPP C
Resultat: coherent
R(A,C) = {TPP, NTPP}
R(C,A) = {TPPI, NTPPI}


## 2. Contradiction directe

On impose d'abord `A TPP B`, puis on essaie d'imposer `A DC B`. Une meme paire ne peut pas etre simultanement incluse et deconnectee.

In [3]:
def direct_contradiction(solver):
    solver.add_constraint("A", "B", {"TPP"})
    solver.add_constraint("A", "B", {"DC"})


run_case("Contradiction directe: A TPP B puis A DC B", direct_contradiction, False)

Contradiction directe: A TPP B puis A DC B
Resultat: incoherent
Diagnostic: Inconsistency detected between A and B


## 3. Contradiction entre une relation et son inverse

Si `A TPP B`, alors l'inverse impose `B TPPI A`. Imposer ensuite `B TPP A` contredit cette relation inverse.

In [4]:
def inverse_contradiction(solver):
    solver.add_constraint("A", "B", {"TPP"})
    solver.add_constraint("B", "A", {"TPP"})


run_case("Contradiction inverse: A TPP B puis B TPP A", inverse_contradiction, False)

Contradiction inverse: A TPP B puis B TPP A
Resultat: incoherent
Diagnostic: Inconsistency detected between B and A


## 4. Contradiction indirecte par propagation

Ici, chaque contrainte semble acceptable localement:

- `A NTPP B`;
- `B NTPP C`;
- `C DC A`.

Mais les deux premieres impliquent que `A NTPP C`, donc `C NTPPI A`. La relation `C DC A` devient impossible.

In [5]:
def hidden_contradiction(solver):
    solver.add_constraint("A", "B", {"NTPP"})
    solver.add_constraint("B", "C", {"NTPP"})
    solver.add_constraint("C", "A", {"DC"})


run_case("Contradiction indirecte: inclusion stricte contre deconnexion", hidden_contradiction, False)

Contradiction indirecte: inclusion stricte contre deconnexion
Resultat: incoherent
Diagnostic: Inconsistency detected between A and C


## 5. Comparaison: retirer la contrainte contradictoire

Si on retire `C DC A`, le reseau redevient coherent et la propagation deduit automatiquement la relation entre `A` et `C`.

In [6]:
def strict_inclusion_chain(solver):
    solver.add_constraint("A", "B", {"NTPP"})
    solver.add_constraint("B", "C", {"NTPP"})


solver_strict = run_case("Cas coherent: A NTPP B, B NTPP C", strict_inclusion_chain, True)
show_pair(solver_strict, "A", "C")

assert solver_strict.R[("A", "C")] == {"NTPP"}
assert solver_strict.R[("C", "A")] == {"NTPPI"}

Cas coherent: A NTPP B, B NTPP C
Resultat: coherent
R(A,C) = {NTPP}
R(C,A) = {NTPPI}


## 6. A retenir

- Une incoherence est detectee quand un domaine devient vide.
- Certaines contradictions sont immediates, par exemple deux relations incompatibles sur la meme paire.
- D'autres contradictions sont cachees et apparaissent seulement apres composition via une troisieme region.
- La propagation PC-2 permet de trouver ces contradictions indirectes.